In [3]:
import pandas as pd
events_df = pd.read_csv(r'C:\Users\днс\Downloads\events.csv', sep=',')
purchase_df = pd.read_csv(r'C:\Users\днс\Downloads\purchase.csv', sep=',')
events_df['start_time'] = pd.to_datetime(events_df['start_time'], errors='coerce')
purchase_df['event_datetime'] = pd.to_datetime(purchase_df['event_datetime'], errors='coerce')

mask1 = events_df['start_time'] >= '2018-01-01'
mask2 = events_df['start_time'] < '2019-01-01'
mask3 = events_df['event_type'] == 'registration'
users_2018 = events_df[mask1 & mask2 & mask3]['user_id'].to_list()
events_df  = events_df[events_df['user_id'].isin(users_2018)]

purchase_df  = purchase_df[purchase_df['user_id'].isin(users_2018)]
purchase_df['event_type'] = 'purchase'
events_df = events_df.rename(columns={"id": "event_id"})
purchase_df = purchase_df.rename(columns={"id": "purchase_id", "event_datetime": "start_time"})
total_events_df = pd.concat([events_df,purchase_df],sort=False)

total_events_df = total_events_df.reset_index(drop=True).sort_values('start_time')

levels_df = total_events_df[total_events_df['selected_level'].notna()]

easy_lvl = levels_df[levels_df["selected_level"] == "easy"]["user_id"].unique()
medium_lvl = levels_df[levels_df["selected_level"] == "medium"]["user_id"].unique()
hard_lvl = levels_df[levels_df["selected_level"] == "hard"]["user_id"].unique()

purchase_easy_df = purchase_df[purchase_df['user_id'].isin(easy_lvl)]
percent_easy = (purchase_easy_df['user_id'].nunique() / len(easy_lvl))
print("Процент оплат среди уровня easy: {:.2%}".format(percent_easy))

purchase_medium_df = purchase_df[purchase_df['user_id'].isin(medium_lvl)]
percent_medium = (purchase_medium_df['user_id'].nunique() / len(medium_lvl))
print("Процент оплат среди уровня medium: {:.2%}".format(percent_medium))

purchase_hard_df = purchase_df[purchase_df['user_id'].isin(hard_lvl)]
percent_hard = (purchase_hard_df['user_id'].nunique() / len(hard_lvl))
print("Процент оплат среди уровня hard: {:.2%}".format(percent_hard))

# Собятия оплат и выбора уровней сложности

events_easy_df = levels_df[levels_df['selected_level'] == 'easy'][['user_id', 'start_time']]
events_easy_df['start_time'] = pd.to_datetime(events_easy_df['start_time'])

purchase_easy_events_df = purchase_df[purchase_df['user_id'].isin(easy_lvl)][['user_id', 'start_time']]
purchase_easy_events_df['start_time'] = pd.to_datetime(purchase_easy_events_df['start_time'])

events_medium_df = levels_df[levels_df['selected_level'] == 'medium'][['user_id', 'start_time']]
events_medium_df['start_time'] = pd.to_datetime(events_medium_df['start_time'])

purchase_medium_events_df = purchase_df[purchase_df['user_id'].isin(medium_lvl)][['user_id', 'start_time']]
purchase_medium_events_df['start_time'] = pd.to_datetime(purchase_medium_events_df['start_time'])

events_hard_df = levels_df[levels_df['selected_level'] == 'hard'][['user_id', 'start_time']]
events_hard_df['start_time'] = pd.to_datetime(events_hard_df['start_time'])

purchase_hard_events_df = purchase_df[purchase_df['user_id'].isin(hard_lvl)][['user_id', 'start_time']]
purchase_hard_events_df['start_time'] = pd.to_datetime(purchase_hard_events_df['start_time'])

Процент оплат среди уровня easy: 7.72%
Процент оплат среди уровня medium: 20.86%
Процент оплат среди уровня hard: 35.39%


Среднее время между событиями оплаты и выбором уровня сложности

In [22]:
easy_choice_df = total_events_df[total_events_df["selected_level"] == "easy"]
print(easy_choice_df["user_id"].value_counts().mean())
easy_choice_df = easy_choice_df[["user_id", "start_time"]].rename(columns={"start_time": "easy_choice"})
purchase_easy_events_df = purchase_easy_events_df.rename(columns={"start_time": "purchase_time_easy"})
merged_lvl = purchase_easy_events_df.merge(easy_choice_df, on="user_id", how="inner")
merged_lvl["timediff"] = (merged_lvl["purchase_time_easy"] - merged_lvl["easy_choice"])
print(merged_lvl["timediff"].mean())
print(merged_lvl["timediff"].describe())

1.0
3 days 14:58:52.941798941
count                          189
mean     3 days 14:58:52.941798941
std      2 days 07:06:35.644097504
min                0 days 00:49:20
25%                1 days 17:18:56
50%                3 days 06:03:50
75%                5 days 06:58:18
max               10 days 18:35:09
Name: timediff, dtype: object


In [23]:
medium_choice_df = total_events_df[total_events_df["selected_level"] == "medium"]
print(easy_choice_df["user_id"].value_counts().mean())
medium_choice_df = medium_choice_df[["user_id", "start_time"]].rename(columns={"start_time": "medium_choice"})
purchase_medium_events_df = purchase_medium_events_df.rename(columns={"start_time": "purchase_time_medium"})
merged_lvl_2 = purchase_medium_events_df.merge(medium_choice_df, on="user_id", how="inner")
merged_lvl_2["timediff"] = (merged_lvl_2["purchase_time_medium"] - merged_lvl_2["medium_choice"])
print(merged_lvl_2["timediff"].mean())
print(merged_lvl_2["timediff"].describe())

1.0
3 days 23:14:13.165118679
count                          969
mean     3 days 23:14:13.165118679
std      2 days 06:18:57.618467109
min                0 days 04:18:12
25%                2 days 01:20:07
50%                3 days 19:53:19
75%                5 days 16:07:19
max               10 days 13:51:01
Name: timediff, dtype: object


In [24]:
hard_choice_df = total_events_df[total_events_df["selected_level"] == "hard"]
print(hard_choice_df["user_id"].value_counts().mean())
hard_choice_df = hard_choice_df[["user_id", "start_time"]].rename(columns={"start_time": "hard_choice"})
purchase_hard_events_df = purchase_hard_events_df.rename(columns={"start_time": "purchase_time_hard"})
merged_lvl_3 = purchase_hard_events_df.merge(hard_choice_df, on="user_id", how="inner")
merged_lvl_3["timediff"] = (merged_lvl_3["purchase_time_hard"] - merged_lvl_3["hard_choice"])
print(merged_lvl_3["timediff"].mean())
print(merged_lvl_3["timediff"].describe())

1.0
3 days 07:20:41.420814479
count                          442
mean     3 days 07:20:41.420814479
std      1 days 21:43:52.953292605
min                0 days 03:26:45
25%         1 days 14:57:23.500000
50%         3 days 03:13:57.500000
75%         4 days 19:16:00.250000
max                8 days 01:18:13
Name: timediff, dtype: object


Среднее время от регистрации до оплаты

In [ ]:
registration_df = total_events_df[total_events_df['event_type'] == 'registration']
registration_df['user_id'].value_counts().mean()
registration_df = registration_df[["user_id", "start_time"]].rename(
    columns={"start_time": "registration_time"})
purchase_df = total_events_df[total_events_df['event_type'] == 'purchase']
purchase_df = purchase_df[["user_id", "start_time"]].rename(columns={"start_time": "purchase_time"})
merged_reg = pd.merge(registration_df, purchase_df, on='user_id', how='inner')
merged_reg['time_to_purchase'] = merged_reg['purchase_time'] - merged_reg['registration_time']
print(merged_reg['time_to_purchase'].mean()) 

4 days 01:01:56.595000


В ходе анализа выявлено, что среднее время принятия решения о покупке примерно одинаково во всех группах около 3–4 дней. В среднем пользователи тратят сопоставимое время на выбор продукта, независимо от сложности. Начальный уровень самый непредсказуемый - здесь есть те, кто покупает многновенно и те, кто думает об этом до 10 дней. Сложный уровень - самые быстрые и уверенные покупатели. Они действуют оперативно и почти одинаково, но таких пользователей меньше всего.
Что делать: напоминать о покупке на 3–4 день, уделять особое внимание группе "medium" и бережно относится к группе "hard". 